# Experiment 0 — pilot run

Goal: **does every method run end-to-end** on every dataset? One fold, no HPO. Use these results to curate Experiment 1's method set.

All logic lives in `src/visualizations/experiment_plots.py`; figures are saved as PDF under `figures/experiment0/` (wiped on each rerun).

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, method_ranking_bars, per_dataset_bars,
    learning_curve, imbalance_curve, metric_boxplots,
    hpo_improvement_bars, runtime_performance_scatter,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment0')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
pd_df  = load_summary(SUMMARY_DIR, experiment='experiment0', task='pd')
try:
    lgd_df = load_summary(SUMMARY_DIR, experiment='experiment0', task='lgd')
except FileNotFoundError:
    lgd_df = None
print(f'PD: {pd_df["method"].nunique()} methods x {pd_df["dataset"].nunique()} datasets')

## Coverage — which (method, dataset) cells produced a result?

In [ ]:
cov = pd_df.groupby('method')['dataset'].nunique().sort_values()
n_ds = pd_df['dataset'].nunique()
print('Methods with missing PD datasets (ran on < all):')
print(cov[cov < n_ds].to_string() if (cov < n_ds).any() else '  none — full coverage')

## Quick performance overview (1 fold — indicative only)

In [ ]:
performance_heatmap(pd_df, 'AUC', task_name='PD', out_dir=FIGURES_DIR / 'pd')
method_ranking_bars(pd_df, 'AUC', task_name='PD', out_dir=FIGURES_DIR / 'pd')

In [ ]:
if lgd_df is not None:
    performance_heatmap(lgd_df, 'R2', task_name='LGD', out_dir=FIGURES_DIR / 'lgd')
    method_ranking_bars(lgd_df, 'R2', task_name='LGD', out_dir=FIGURES_DIR / 'lgd')

## Cost — how expensive is each method?

In [ ]:
runtime_performance_scatter(pd_df, 'AUC', task_name='PD', out_dir=FIGURES_DIR / 'pd')
if lgd_df is not None:
    runtime_performance_scatter(lgd_df, 'R2', task_name='LGD', out_dir=FIGURES_DIR / 'lgd')